# Table of Contents
## 1. Imports 'cleaning.csv' into 'cleaning' dataframe and creates 'market_close_16:15' column showing where the market price closed for each day
## 2. Does the same at 1. but creates 'market_reg_close_16:00'
## 3. Removes from 'cleaning' df 'initial_direction'== 'equal' rows and assign those rows to 'equal_rows'. Saves dataframes to csv files
## 4. Creates dataframe of just rows where 'time'== '09:30:00' and saves to csv file

## 1.

In [2]:

def check_close_bar(mapping_index, label):
    """Warn if any trading date in `cleaning` has no bar at the given close time.
    A missing bar means market_close_{label} will be NaN for all rows of that date.
    """
    all_dates = set(cleaning['date'].unique())
    missing = all_dates - set(mapping_index)
    if missing:
        print(f"WARNING: {len(missing)} date(s) missing {label} close bar:")
        for d in sorted(missing):
            print(f"  {d}")
    else:
        print(f"OK: all {len(all_dates)} date(s) have a {label} close bar")

In [3]:
import pandas as pd

cleaning= pd.read_csv('cleaning.csv')

# 1. Ensure 'date' is in a standardized date format (if it's still a string).
#    If it's already datetime or if you only need the date string, you can skip this step.
cleaning['date'] = pd.to_datetime(cleaning['date'])

# 2. Create a mapping from date -> candle_open_price for rows where confirm_time == '16:15:00'.
#    Drop duplicates in case multiple rows for the same date have confirm_time == '16:15:00'.
mapping = (
    cleaning.loc[cleaning['confirm_time'] == '16:15:00', ['date', 'candle_open_price']]
      .drop_duplicates('date')
      .set_index('date')['candle_open_price']
)

# 3. Create the new column 'market_close_16:15' by mapping each row's date to the above Series.
#    This will fill the column with NaN if there's no matching date in the mapping.
cleaning['market_close_16:15'] = cleaning['date'].map(mapping)

# Now df has a new column 'market_close_16:15' that, for each date,
# holds the candle_open_price from the row where confirm_time == '16:15:00' for that date.
print(cleaning.head(75))  # Check the first 20 rows to verify

         date  candle_open_price  candle_high_price  candle_low_price  \
0  2024-03-11            5188.00            5188.25           5185.75   
1  2024-03-11            5186.50            5187.50           5186.00   
2  2024-03-11            5184.50            5184.75           5183.00   
3  2024-03-11            5182.75            5183.25           5182.25   
4  2024-03-11            5179.50            5179.50           5176.50   
..        ...                ...                ...               ...   
70 2024-03-12            5207.75            5208.50           5205.75   
71 2024-03-12            5202.25            5203.00           5199.00   
72 2024-03-12            5206.25            5209.75           5205.25   
73 2024-03-12            5208.00            5209.25           5205.00   
74 2024-03-12            5208.00            5210.00           5206.00   

    candle_close_price  volume  average  bar_count      time confirm_time  \
0              5186.00    -1.0     -1.0       

In [4]:
check_close_bar(mapping.index, '16:15')

  2024-05-27 00:00:00
  2024-06-17 00:00:00
  2024-06-18 00:00:00
  2024-06-19 00:00:00
  2024-07-04 00:00:00
  2024-07-16 00:00:00
  2024-07-30 00:00:00
  2024-08-13 00:00:00
  2024-08-27 00:00:00
  2024-09-02 00:00:00
  2024-11-28 00:00:00
  2024-12-16 00:00:00
  2024-12-17 00:00:00
  2025-01-20 00:00:00
  2025-02-17 00:00:00
  2025-05-26 00:00:00
  2025-06-19 00:00:00
  2025-07-03 00:00:00
  2025-07-04 00:00:00
  2025-09-01 00:00:00
  2025-11-27 00:00:00
  2025-11-28 00:00:00
  2025-12-24 00:00:00
  2026-01-19 00:00:00
  2026-02-16 00:00:00


In [5]:
cleaning.head()

,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,month,day,day_of_week,market_open_09:30,initial_direction,above,below,market_open,contract_location,market_close_16:15
0,2024-03-11,5188.00,5188.25,5185.75,5186.00,-1.0,-1.0,-1.0,07:00:00,07:00:00,...,3,11,Monday,NaN,equal,5181.0,5169.0,5177.75,NaN,5188.0
1,2024-03-11,5186.50,5187.50,5186.00,5186.25,-1.0,-1.0,-1.0,07:15:00,07:15:00,...,3,11,Monday,NaN,equal,5181.0,5169.0,5177.75,NaN,5188.0
2,2024-03-11,5184.50,5184.75,5183.00,5183.50,-1.0,-1.0,-1.0,07:30:00,07:30:00,...,3,11,Monday,NaN,equal,5181.0,5169.0,5177.75,NaN,5188.0
3,2024-03-11,5182.75,5183.25,5182.25,5182.75,-1.0,-1.0,-1.0,07:45:00,07:45:00,...,3,11,Monday,NaN,equal,5181.0,5169.0,5177.75,NaN,5188.0
4,2024-03-11,5179.50,5179.50,5176.50,5177.50,-1.0,-1.0,-1.0,08:00:00,08:00:00,...,3,11,Monday,NaN,equal,5181.0,5169.0,5177.75,NaN,5188.0


## 2.

In [6]:
import pandas as pd

# 1. Ensure 'date' is in a standardized date format (if it's still a string).
#    If it's already datetime or if you only need the date string, you can skip this step.
cleaning['date'] = pd.to_datetime(cleaning['date'])

# 2. Create a mapping from date -> candle_open_price for rows where confirm_time == '16:00:00'.
#    Drop duplicates in case multiple rows for the same date have confirm_time == '16:00:00'.
mapping = (
    cleaning.loc[cleaning['confirm_time'] == '16:00:00', ['date', 'candle_open_price']]
      .drop_duplicates('date')
      .set_index('date')['candle_open_price']
)

# 3. Create the new column 'market_close_16:00' by mapping each row's date to the above Series.
#    This will fill the column with NaN if there's no matching date in the mapping.
cleaning['market_reg_close_16:00'] = cleaning['date'].map(mapping)

# Now df has a new column 'market_close_16:00' that, for each date,
# holds the candle_open_price from the row where confirm_time == '16:00:00' for that date.
print(cleaning.head(75))  # Check the first 20 rows to verify

cleaning.to_csv('cleaning.csv', index= False)


         date  candle_open_price  candle_high_price  candle_low_price  \
0  2024-03-11            5188.00            5188.25           5185.75   
1  2024-03-11            5186.50            5187.50           5186.00   
2  2024-03-11            5184.50            5184.75           5183.00   
3  2024-03-11            5182.75            5183.25           5182.25   
4  2024-03-11            5179.50            5179.50           5176.50   
..        ...                ...                ...               ...   
70 2024-03-12            5207.75            5208.50           5205.75   
71 2024-03-12            5202.25            5203.00           5199.00   
72 2024-03-12            5206.25            5209.75           5205.25   
73 2024-03-12            5208.00            5209.25           5205.00   
74 2024-03-12            5208.00            5210.00           5206.00   

    candle_close_price  volume  average  bar_count      time confirm_time  \
0              5186.00    -1.0     -1.0       

In [7]:
check_close_bar(mapping.index, '16:00')

  2024-05-27 00:00:00
  2024-06-17 00:00:00
  2024-06-18 00:00:00
  2024-06-19 00:00:00
  2024-07-04 00:00:00
  2024-07-16 00:00:00
  2024-07-30 00:00:00
  2024-08-13 00:00:00
  2024-08-27 00:00:00
  2024-09-02 00:00:00
  2024-11-28 00:00:00
  2024-12-16 00:00:00
  2024-12-17 00:00:00
  2025-01-20 00:00:00
  2025-02-17 00:00:00
  2025-05-26 00:00:00
  2025-06-19 00:00:00
  2025-07-03 00:00:00
  2025-07-04 00:00:00
  2025-09-01 00:00:00
  2025-11-27 00:00:00
  2025-11-28 00:00:00
  2025-12-24 00:00:00
  2026-01-19 00:00:00
  2026-02-16 00:00:00


In [166]:
cleaning.head()

,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,month,day,day_of_week,market_open_09:30,initial_direction_open,above,below,contract_location,market_close_16:15,market_close_16:00
0,2022-12-12,3976.25,3981.50,3974.75,3980.00,13850.0,3977.425,4176,09:30:00,09:30:00,...,12,12,Monday,3976.25,equal,3980.0,3968.0,NaN,4020.25,4025.75
1,2022-12-12,3980.00,3980.50,3976.00,3977.50,8373.0,3977.900,2593,09:31:00,09:31:00,...,12,12,Monday,3976.25,above,3980.0,3968.0,3968.0,4020.25,4025.75
2,2022-12-12,3977.25,3979.25,3975.25,3978.50,5649.0,3977.150,1895,09:32:00,09:32:00,...,12,12,Monday,3976.25,above,3980.0,3968.0,3968.0,4020.25,4025.75
3,2022-12-12,3978.50,3979.75,3975.50,3978.75,5903.0,3978.200,2027,09:33:00,09:33:00,...,12,12,Monday,3976.25,above,3980.0,3968.0,3968.0,4020.25,4025.75
4,2022-12-12,3978.50,3979.00,3973.75,3974.50,5902.0,3975.675,1996,09:34:00,09:34:00,...,12,12,Monday,3976.25,above,3980.0,3968.0,3968.0,4020.25,4025.75


## 3.

In [8]:
equal_rows = cleaning[cleaning['initial_direction'] == 'equal']
equal_rows.to_csv('equal_rows.csv', index=False)
print(equal_rows.head())

clean = cleaning[cleaning['initial_direction'] != 'equal']
clean.to_csv('clean.csv', index=False)
print(clean.head())

        date  candle_open_price  candle_high_price  candle_low_price  \
0 2024-03-11            5188.00            5188.25           5185.75   
1 2024-03-11            5186.50            5187.50           5186.00   
2 2024-03-11            5184.50            5184.75           5183.00   
3 2024-03-11            5182.75            5183.25           5182.25   
4 2024-03-11            5179.50            5179.50           5176.50   

   candle_close_price  volume  average  bar_count      time confirm_time  ...  \
0             5186.00    -1.0     -1.0       -1.0  07:00:00     07:00:00  ...   
1             5186.25    -1.0     -1.0       -1.0  07:15:00     07:15:00  ...   
2             5183.50    -1.0     -1.0       -1.0  07:30:00     07:30:00  ...   
3             5182.75    -1.0     -1.0       -1.0  07:45:00     07:45:00  ...   
4             5177.50    -1.0     -1.0       -1.0  08:00:00     08:00:00  ...   

   day  day_of_week  market_open_09:30 initial_direction   above   below  \
0   

## 4.

In [9]:
open_rows = equal_rows[equal_rows['time'] == '09:30:00']

open_rows = open_rows[['date', 'market_open_09:30']]

open_rows.to_csv('open_rows.csv', index=False)